# Two-tone coherence: does onset asynchrony break binding?

**Elhilali et al. (2009, _Neuron_)** argue that two sounds fuse into one stream when their
channels are temporally **coherent**, and split when they are not. Their Figure 8B makes that
quantitative for the simplest scene there is — two tones — by sweeping the onset asynchrony
ΔT from exact synchrony to exact alternation and plotting the model's segregation index
λ₂/λ₁, which climbs from 0.01 to 0.93.

**That curve has never been measured behaviourally.** Their own psychophysics (Figure 2)
compares only two states, and does so through a tempo difference rather than a fixed lag.

This notebook lays out a task that measures it, using their asynchrony-detection paradigm with
ΔT as a parametric variable. Everything here runs from the repository; nothing is hand-drawn.

> **There is no listener data in this notebook.** The analysis section runs a *simulated*
> observer, and says so on every figure it makes.

In [ ]:
#@title setup
import sys, subprocess, importlib.util, json, math
from pathlib import Path
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
if importlib.util.find_spec('tcoh') is None:
    if not Path('SeqSFG_task').exists():
        subprocess.run(['git','clone','-q',REPO], check=True)
    sys.path.insert(0,'SeqSFG_task')
get_ipython().run_line_magic('matplotlib','inline')
import numpy as np, matplotlib
matplotlib.rcParams['figure.dpi']=120
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown, HTML
from tcoh.config import DEFAULT, validate, conditions
from tcoh import model as M, stimulus as S, plots as P, verify as V
CFG = DEFAULT; D = validate(CFG); CONDS = conditions(CFG)
def head(s): display(Markdown(s))
head(f"**A {CFG.f_a_hz:.0f} Hz, B {D.f_b_hz:.0f} Hz** ({CFG.df_semitones:g} semitones, "
     f"{D.erbs_apart:.1f} ERB apart) &nbsp;|&nbsp; {CFG.tone_ms:g} ms tones, {CFG.soa_ms:g} ms SOA, "
     f"{CFG.n_tones} per channel &nbsp;|&nbsp; config `{CFG.hash()}`")

---
## 1. The task

Two sequences per trial. They are identical **except that in one of them the last high tone is
displaced by ±δ**. Which one? An adaptive staircase tracks δ.

If the two tones are heard as one object, displacing one changes that object and is detectable
at a few milliseconds. If they are heard as two streams, the listener has only the high tone's
own rhythm to go on, and thresholds are an order of magnitude worse. **How far along that range
a listener sits is a measure of whether the tones are bound.**

In [ ]:
#@title the stimulus at each asynchrony
P.schematic(CFG); plt.show()
head("Only the outlined tone moves. The low tone never moves, and the high tone's grid is the "
     "same in every row — so the information in either tone *alone* is the same in every "
     "condition, and only the **relation** between them varies with ΔT.")

---
## 2. What it sounds like

Below, one trial per asynchrony, with the shift set to 25 ms — well above threshold, so it is
audible. **In every clip the FIRST of the two sounds is the one with the displaced tone.**

Listen for what changes as you go down the list: at ΔT = 0% the pair is a single chord and the
shift makes it "split"; by ΔT = 100% the tones have become two separate trains and you are
judging a rhythm instead.

In [ ]:
#@title audio, one per asynchrony
rng = np.random.default_rng(4)
for c in [x for x in CONDS if x.a_kind=='coherent']:
    tr = S.build_trial(CFG, c, 25.0, np.random.default_rng(4), target_position=1, direction=+1)
    x  = S.render_trial(CFG, tr, D)
    head(f"**ΔT = {c.lag_pct:g}%** &nbsp; (lag {CFG.lag_ms(c.lag_pct):.1f} ms; "
         f"model λ₂/λ₁ = {D.model_by_condition[c.name]:.3f})")
    display(Audio(x, rate=CFG.sample_rate, normalize=False))

In [ ]:
#@title audio: the two ends of the scale
for name, label in (('b_only','the ceiling: low tone switched off entirely'),
                    ('scr_0','the control: low tone present and synchronous at the end, but its earlier tones scrambled')):
    c = next(x for x in CONDS if x.name==name)
    tr = S.build_trial(CFG, c, 25.0, np.random.default_rng(4), target_position=1, direction=+1)
    head(f"**{name}** — {label}")
    display(Audio(S.render_trial(CFG, tr, D), rate=CFG.sample_rate, normalize=False))

---
## 3. The model, and one thing it decided

`tcoh/model.py` re-implements the paper's coherence analysis for two channels, so the
prediction is for *our* stimulus rather than read off their figure with a ruler.

Taken as bare arithmetic the published equation does **not** reproduce the published numbers:
two perfectly alternating channels are *anti*-correlated, not uncorrelated. Reading step 2 as
the **coincidence detection** the paper's own text describes — a rectified product — reproduces
both stated values and the stated monotonicity.

In [ ]:
#@title check against the published Figure 8
r = M.reproduce_figure8()
rows = ["| reading | ΔT = 100% | ΔT = 0% | monotone |","|---|---|---|---|"]
for k,v in r['readings'].items():
    rows.append(f"| {'**'+k+'**' if k==r['best'] else k} | {v['alternating']:.3f} | "
                f"{v['synchronous']:.3f} | {v['monotone']} |")
rows.append(f"| _published_ | _{M.PUBLISHED['alternating']}_ | _{M.PUBLISHED['synchronous']}_ | _yes_ |")
head("\n".join(rows))
head(f"This package uses **{r['best']}**. It is an inference about an under-specified method, "
     "and is flagged as one everywhere it matters.")

In [ ]:
#@title what the model sees
P.envelopes(CFG); plt.show()

In [ ]:
#@title ΔT is only an ordered axis at a 50% duty cycle
scan = M.duty_cycle_scan(soa_ms=CFG.soa_ms, n_tones=CFG.n_tones)
rows=["| tone / SOA | monotone in ΔT? | peak |","|---|---|---|"]
for duty,v in scan.items():
    rows.append(f"| {'**'+format(duty,'.3f')+'**' if abs(duty-CFG.duty)<1e-9 else format(duty,'.3f')} | "
                f"{'yes' if v['monotone'] else '**no**'} | ΔT = {v['argmax_pct']:.0f}% |")
head("\n".join(rows))
head(f"With a silent gap inside each channel the predicted index **peaks near ΔT = 75%** and "
     "falls again at full alternation — so a monotone behavioural result would confirm nothing. "
     f"This is why `tone_ms` is fixed at exactly half of `soa_ms` ({CFG.tone_ms:g} / {CFG.soa_ms:g}) "
     "and the validator refuses anything else.")

In [ ]:
#@title the prediction, before any data
P.prediction(CFG); plt.show()
head("The band is the spread across eight defensible readings of the filter bank. "
     "**The ordering is robust; the heights are not** — which is why the shape of the curve is "
     "not treated as evidence.")

---
## 4. Why the controls decide it, and H1 does not

Three accounts all predict that thresholds rise with ΔT:

- **temporal coherence** — the tones stop being one object, so the low tone stops being usable;
- **interval discrimination (Weber)** — the A–B interval the listener judges grows with ΔT, and
  judging a change in a longer interval is harder, with no streaming involved;
- **local acoustic overlap** — the two tones simply overlap less.

So "threshold rises with ΔT" settles nothing. The controls do. Both hold the **final A–B
interval exactly** — the last low tone is where the coherent condition puts it — and remove
only the low tone's *sequence*. The two rival accounts depend only on the final pair, so both
predict **no difference at any ΔT**. The coherence account predicts a large difference at
synchrony that vanishes at alternation. Different shapes.

In [ ]:
#@title the predicted interaction
coh = {c.lag_pct: D.model_by_condition[c.name] for c in CONDS if c.a_kind=='coherent'}
ctl = {c.lag_pct: D.model_by_condition[c.name] for c in CONDS if c.a_kind=='scrambled'}
sh  = sorted(set(coh)&set(ctl))
rows=["| ΔT | coherent | scrambled control | difference |","|---|---|---|---|"]
for p in sh: rows.append(f"| {p:g}% | {coh[p]:.3f} | {ctl[p]:.3f} | **{ctl[p]-coh[p]:+.2f}** |")
head("\n".join(rows))
head("Positive at synchrony, falling to negative at alternation. A pedestal or local-overlap "
     "account predicts a column of zeros. That contrast is the whole experiment.")

---
## 5. Everything checkable without a listener

22 checks. The three that carry the argument are the invariants: within a trial the two
intervals are bit-identical except for one tone's position; the low tone never moves, so it
carries no information about the answer; and the high tone's grid is identical in every
condition, so its own cue cannot vary with ΔT.

In [ ]:
#@title run the battery
bat = V.run_battery(CFG, quick=True)
for section, checks in bat.items():
    head(f"**{section}**")
    for c in checks:
        head(f"- {'✅' if c.passed else '❌'} {c.name}  \n  <small>{c.detail}</small>")

---
## 6. Power, measured rather than assumed

The whole pipeline — staircase, design, analysis — run against simulated listeners generated
by the hypothesis and by each rival. **Read the middle row**: the Weber rival produces the
"threshold rises with ΔT" result on *every single* simulated session.

| generating truth | H1 fires | H2 fires |
|---|---|---|
| coherence (the hypothesis) | 100% | **87%** |
| pedestal (the Weber rival) | **100%** | 4% |
| null (nothing depends on ΔT) | 6% | 3% |

A shortened design — control at three ΔT levels instead of five — keeps 99% power for H1 and
drops to **9%** for H2. It is a screen, not a test, and the validator says so on every run.

---
## 7. A worked analysis — on a SIMULATED listener

Everything below comes from an observer generated by `tcoh.observer`, not from a person. It is
here to show what the analysis produces and that it discriminates, not to claim a result.

In [ ]:
#@title simulate a session and analyse it
import tempfile
from tcoh.runner import Runner
from tcoh.analysis import analyse, coherence_index, interaction_test, load, thresholds
tmp = Path(tempfile.mkdtemp())
r = Runner(CFG, tmp, audio=False, auto='coherence', seed=11)
sdir = r.run(code='SIM')
rows, metas = load([sdir])
res = thresholds(rows, CFG)
idx = coherence_index(res, CFG, n_boot=1200)
ctl = coherence_index(res, CFG, n_boot=1200, a_kind='scrambled')
inter = interaction_test(res, CFG, n_boot=1200)
P.curve(CFG, idx, ctl, simulated=True); plt.show()
P.controls(CFG, inter, simulated=True); plt.show()

In [ ]:
#@title the same session under the rival, to show the analysis discriminates
r2 = Runner(CFG, Path(tempfile.mkdtemp()), audio=False, auto='pedestal', seed=11)
s2 = r2.run(code='SIM')
i2 = interaction_test(thresholds(load([s2])[0], CFG), CFG, n_boot=1200)
rows=["| generating truth | H2 slope | permutation p | verdict |","|---|---|---|---|"]
for nm,t in (('coherence',inter),('pedestal (rival)',i2)):
    rows.append(f"| {nm} | {t['slope_per_pct']*100:+.2f} | {t['p_slope_negative']:.4f} | "
                f"{'**consistent with coherence**' if t['p_slope_negative']<0.05 else 'not resolved'} |")
head("\n".join(rows))

In [ ]:
#@title the full text report
print(analyse([sdir], n_boot=1200))

---
## 8. What this cannot establish

- **It cannot confirm the model's shape.** Over these ΔT levels the predicted curve correlates
  with a straight line at r = 0.988. No realistic amount of data separates them, and the report
  says so where it prints the comparison.
- **Onset lag and acoustic overlap are perfectly confounded** at a 50% duty cycle: ΔT% is
  exactly 100 × (1 − overlap fraction). Breaking that needs a duty-cycle manipulation — a
  separate experiment, which the config already supports.
- **The controls are imperfect in different directions.** The scrambled control matches tone
  count and energy but its own coherence is not flat across ΔT; the pair-only control has no
  low-tone sequence at all but holds five fewer tones. The argument rests on their agreeing.
- **κ is normalised by two conditions measured in the same session**, so a bad floor or ceiling
  moves the whole curve. Both are printed in milliseconds beside it.
- **One listener generalises to one listener.**

The pre-registration (`tcoh/PREREGISTRATION.md`) states the hypotheses, the decision rules and
the exclusion criteria in full, and was written before any listener was run.